# ML In-Class Exam — Cheatsheet
### Binary Classification with scikit-learn | All 5 Tasks
> Replace `TARGET`, `FEATURE_X`, column names, and `student_id` with your actual values. Swap model names in markdown cells to match what you pick.

In [ ]:
# ── BOILERPLATE: Run this first every time ──────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

np.random.seed(419414)  # your student ID — change this

---
## Task 1 — Exploratory Data Analysis

In [ ]:
# TASK 1.1 — Display the shape of the dataset and the first few rows
df = pd.read_csv('data.csv')  # change filename

print('Dataset shape:', df.shape)
df.head()

In [ ]:
# TASK 1.2 — Check for missing values across all columns and report findings
df.info()  # shows dtypes and non-null counts

In [ ]:
# TASK 1.2 (continued) — Explicit null count per column
df.isnull().sum()

In [ ]:
# TASK 1.2 (continued) — If zeros are not plausible (e.g. medical data), flag them too
# SKIP THIS CELL if zeros are valid in your dataset
cols_with_invalid_zeros = ['Feature1', 'Feature2', 'Feature3']  # change to your columns
print('Zero counts (potential missing data):')
print((df[cols_with_invalid_zeros] == 0).sum())

In [ ]:
# TASK 1.3 — Plot the class distribution of the target column and comment on what you observe
sns.countplot(data=df, x='Target')  # change 'Target' to your target column name
plt.title('Class Distribution of Target')
plt.show()

print(df['Target'].value_counts())

In [ ]:
# TASK 1.4 — Plot boxplots for at least 4 features comparing the two classes
# Identify which features best separate the classes and which appear problematic
features = ['Feature1', 'Feature2', 'Feature3', 'Feature4']  # pick at least 4

plt.figure(figsize=(12, 8))
for i, col in enumerate(features, 1):
    plt.subplot(2, 2, i)
    sns.boxplot(data=df, x='Target', y=col)  # change 'Target'
    plt.title(f'{col} by Target')
plt.tight_layout()
plt.show()

### EDA Summary
<!-- TASK 1.5 — Write 3–5 sentences covering: class balance, data quality issues, most informative features -->

The dataset contains [N] records with a [balanced / moderately imbalanced / heavily imbalanced] class distribution — [X] positive cases vs [Y] negative cases. [Comment on any data quality issues identified, e.g. missing values, invalid zeros encoded as such, skewed distributions.] From the visualisations, [Feature A] and [Feature B] show the clearest separation between the two classes, suggesting they will be the most informative predictors. [Feature C] appears noisy with significant overlap between classes, which may limit its individual usefulness. Overall the dataset is usable but requires [describe preprocessing steps] before modelling.

---
## Task 2 — Data Preparation

### Data Quality Justification
<!-- TASK 2.1 — Justify your preprocessing choices in a markdown cell -->

**If imputing zeros (e.g. medical dataset):** Columns [X, Y, Z] contained zeros that are not physiologically plausible and were replaced with NaN. Median imputation was used because these features are skewed and the median is more robust to outliers than the mean. This preserves the full dataset size while correcting invalid entries.

**If dropping columns (e.g. IDs, names, high-cardinality strings):** [ID / Name / TypeCol] were dropped before modelling. [ID] is an arbitrary identifier with no predictive value; [Name] is a free-text string that cannot be used as a numeric feature; [TypeCol] has [X] missing values and many unique categories that would add high dimensionality for limited benefit. The remaining features are all numeric and require no further imputation.

In [ ]:
# TASK 2.1 — Option A: Replace invalid zeros with NaN and impute with median
# Use when zeros are not plausible (e.g. Glucose=0, BMI=0 in medical data)
invalid_zero_cols = ['Feature1', 'Feature2', 'Feature3']  # change to your columns
df[invalid_zero_cols] = df[invalid_zero_cols].replace(0, np.nan)

imputer = SimpleImputer(strategy='median')
df[invalid_zero_cols] = imputer.fit_transform(df[invalid_zero_cols])

In [ ]:
# TASK 2.1 — Option B: Drop columns that are not suitable as features
# Use when dataset has ID columns, name strings, or irrelevant categoricals
df_model = df.drop(columns=['ID', 'Name', 'CatCol1', 'CatCol2'])  # change to your columns

In [ ]:
# TASK 2.2 / 2.3 — Separate features and target, then do 60/20/20 stratified split
# Use stratification on the target column and your student ID as the random seed
X = df_model.drop('Target', axis=1)  # change 'Target' / use df if no columns dropped
y = df_model['Target'].astype(int)   # change 'Target' — .astype(int) handles bool targets

student_id = 419414  # ← change to your actual student ID

# First split: 60% train, 40% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=student_id
)

# Second split: 20% val, 20% test (50% of the 40% temp)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=student_id
)

# Confirm shapes
X_train.shape, X_val.shape, X_test.shape

In [ ]:
# TASK 2.2 — Apply feature scaling (fit on training data ONLY, then transform val and test)
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

---
## Task 3 — Model Training with Cross-Validation
### Model A — Logistic Regression

#### Why Logistic Regression?
<!-- TASK 3 Model A.1 — State which classifier and why it is a reasonable choice -->
Logistic Regression is a strong baseline choice because [features X and Y] show clear linear separation between the two classes in the boxplots. It is interpretable, computationally efficient, and performs well when features are scaled — which has already been applied. It also provides probabilistic outputs, which is useful for understanding prediction confidence when dealing with potentially imbalanced classes.

In [ ]:
# TASK 3 Model A.2 — Define parameter grid and run cross-validated hyperparameter search
log_reg = LogisticRegression(max_iter=1000)

param_grid_lr = {
    'C': [0.01, 0.1, 1, 10],           # regularisation strength
    'solver': ['liblinear', 'lbfgs']    # optimisation algorithm
}

grid_lr = GridSearchCV(
    estimator=log_reg,
    param_grid=param_grid_lr,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_lr.fit(X_train_scaled, y_train)

In [ ]:
# TASK 3 Model A.3 — Report the best parameters found and the mean CV score
print('Model A — Logistic Regression')
print('Best Parameters:', grid_lr.best_params_)
print('Mean CV Score:', grid_lr.best_score_)

### Model B — k-Nearest Neighbours

#### Why KNN?
<!-- TASK 3 Model B.1 — State which classifier and why it is a reasonable choice -->
KNN is a non-parametric classifier that makes no assumptions about the underlying data distribution. It can capture non-linear decision boundaries, which is useful when the relationship between features and the target class is not strictly linear. KNN benefits directly from the feature scaling already applied, and tuning the number of neighbours and distance metric allows it to adapt well to the size and characteristics of the dataset.

In [ ]:
# TASK 3 Model B.2 — Define parameter grid and run cross-validated hyperparameter search
knn = KNeighborsClassifier()

param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9, 11],              # number of neighbours
    'weights': ['uniform', 'distance'],            # weighting scheme
    'metric': ['euclidean', 'manhattan']           # distance metric
}

grid_knn = GridSearchCV(
    estimator=knn,
    param_grid=param_grid_knn,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_knn.fit(X_train_scaled, y_train)

In [ ]:
# TASK 3 Model B.3 — Report the best parameters found and the mean CV score
print('Model B — KNN')
print('Best Parameters:', grid_knn.best_params_)
print('Mean CV Score:', grid_knn.best_score_)

### Model C — Random Forest

#### Why Random Forest?
<!-- TASK 3 Model C.1 — State which classifier and why it is a reasonable choice -->
Random Forest is a strong choice because it handles non-linear relationships and naturally models interactions between features. It is robust to noise, provides built-in feature importance, and — unlike KNN and Logistic Regression — does not require feature scaling. Averaging predictions across many decorrelated trees reduces the risk of overfitting, making it well-suited to smaller datasets with potential class imbalance.

In [ ]:
# TASK 3 Model C.2 — Define parameter grid and run cross-validated hyperparameter search
# NOTE: Random Forest does NOT need scaled data — use X_train, not X_train_scaled
rf = RandomForestClassifier(random_state=student_id)

param_grid_rf = {
    'n_estimators': [100, 200, 300],               # number of trees
    'max_depth': [None, 5, 10, 20],                # maximum tree depth
    'min_samples_split': [2, 5, 10]                # minimum samples to split a node
}

grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid_rf,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)  # ← unscaled data for RF

In [ ]:
# TASK 3 Model C.3 — Report the best parameters found and the mean CV score
print('Model C — Random Forest')
print('Best Parameters:', grid_rf.best_params_)
print('Mean CV Score:', grid_rf.best_score_)

---
## Task 4 — Model Comparison on Validation Set

In [ ]:
# TASK 4.1 / 4.2 — Evaluate all three best models on the validation set
# Report Accuracy and F1 score for each model
best_lr  = grid_lr.best_estimator_
best_knn = grid_knn.best_estimator_
best_rf  = grid_rf.best_estimator_

pred_lr  = best_lr.predict(X_val_scaled)   # LR uses scaled data
pred_knn = best_knn.predict(X_val_scaled)  # KNN uses scaled data
pred_rf  = best_rf.predict(X_val)          # RF uses unscaled data

results = {
    'Model': ['Logistic Regression', 'KNN', 'Random Forest'],
    'Accuracy': [
        accuracy_score(y_val, pred_lr),
        accuracy_score(y_val, pred_knn),
        accuracy_score(y_val, pred_rf)
    ],
    'F1 Score': [
        f1_score(y_val, pred_lr),
        f1_score(y_val, pred_knn),
        f1_score(y_val, pred_rf)
    ]
}

In [ ]:
# TASK 4.2 — Present results clearly as a DataFrame
results_df = pd.DataFrame(results)
results_df

### Model Comparison Summary
<!-- TASK 4.3 — Write 3–5 sentences: which model performs best and why, consider dataset size/balance/features -->
<!-- If the exam specifically asks about accuracy vs F1 (happens with imbalanced datasets), include the second paragraph -->

[Best model] outperforms the other two on the validation set, achieving the highest accuracy and F1 score. This is likely because [reason — e.g. it handles non-linear relationships / it is robust to noise and class imbalance / the features show strong linear separation]. [Weakest model] performs worst, likely because [reason — e.g. its linear decision boundary is too restrictive / it is sensitive to noisy features]. Overall [best model] appears to generalise best given the dataset's size, class balance, and feature characteristics.

**If the dataset is imbalanced:** Given the class imbalance ([majority class count] vs [minority class count]), F1 score is the more informative metric here. A model that simply predicted the majority class for every sample would achieve approximately [majority / total]% accuracy while correctly identifying zero minority cases — so accuracy alone is misleading. F1 score penalises both false positives and false negatives, making it a better indicator of whether the model is actually learning to detect the minority class.

---
## Task 5 — Final Evaluation on Test Set
### Final Model Selection
<!-- TASK 5.1 — State which model you are selecting and justify in 2–3 sentences -->

I am selecting the Random Forest classifier as my final model. It achieved the highest accuracy and F1 score on the validation set, indicating better generalisation compared to Logistic Regression and KNN. Its ability to model non-linear feature interactions and handle class imbalance without requiring feature scaling makes it the most principled choice for this dataset.

In [ ]:
# TASK 5.2 — Retrain chosen model on combined training + validation set
# using the best hyperparameters found in Task 3
X_train_final = np.vstack([X_train, X_val])     # use X_train_scaled + X_val_scaled for LR/KNN
y_train_final = np.hstack([y_train, y_val])

best_rf_params = grid_rf.best_params_

final_rf = RandomForestClassifier(
    **best_rf_params,
    random_state=student_id
)

final_rf.fit(X_train_final, y_train_final)

In [ ]:
# TASK 5.3 — Evaluate the final model on the test set and print the full classification report
test_pred = final_rf.predict(X_test)  # use X_test_scaled for LR or KNN

print('Final Model — Random Forest')
print(classification_report(y_test, test_pred))

In [ ]:
# TASK 5.4 — Plot a confusion matrix heatmap for the test set predictions
cm = confusion_matrix(y_test, test_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix — Final Random Forest Model')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

### Final Model Interpretation
<!-- TASK 5.5 — Write 3–5 sentences: error patterns, class imbalance effects, deployment suitability -->

The final model performs well on the [majority/negative] class, achieving high precision and recall, but shows lower recall for the [minority/positive] class where [X] out of [Y] true cases are missed. This asymmetry is a direct consequence of class imbalance — with fewer [minority class] examples available during training, the model has limited patterns to learn from for that class. The confusion matrix confirms this, showing most errors are false negatives on the minority class rather than false positives. [Comment on whether this is acceptable for the domain, e.g. in a medical context missing a positive case carries a high cost.] While the model shows promise, deploying it in a real-world setting would require further validation and potentially techniques such as SMOTE, class weighting, or threshold tuning to better handle the class imbalance.